# Download, transform, and replace Supabase dataset

This is the manual data-publication workflow. It downloads the configured Amazon
review subset and matching metadata with the datasets library, transforms them in
memory, and publishes the resulting User, Item, and Review tables directly.
No raw or transformed Parquet/CSV files are exported or read back locally.
The datasets library may still use its normal download/Arrow cache.

**Running the final publishing cell intentionally deletes all existing rows in
User, Item, and Review and replaces them with this download.** Table schemas,
indexes, ModelRegistry records, and model Storage artifacts are not replaced.
The configured source category/percentage becomes the entire replacement dataset;
this is not an append or upsert operation. Do not run publication while the app
is serving traffic that requires a consistent dataset.

Publication uses separate Supabase API requests, not one database transaction.
Input validation/serialization happens before deletion, but a network/database
failure during replacement can leave empty or partially populated tables. Fix
the error and rerun the publishing cell to replace all three tables again.


### Imports and source configuration

Use the project kernel with dependencies installed. Set SUPABASE_URL and
SUPBASE_SECRET_KEY in the server-side .env/environment. The key must be allowed
to delete and insert all rows in the three tables. Source settings come from
config/ml_config.py; credentials are never displayed.


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / "config" / "ml_config.py").is_file():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config.ml_config import DATASET_NAME, DATASET_CATEGORY, DATASET_SPLIT_PERCENTAGE
from data_loader import download_data, transform_data
from utils.supabase_utils import replace_data_in_supabase

print({"dataset": DATASET_NAME, "category": DATASET_CATEGORY,
       "review_prefix_percentage": DATASET_SPLIT_PERCENTAGE})


### Download directly into DataFrames

Reviews use the configured prefix percentage; matching category metadata is
loaded in full. This prefix is not a random or representative sample.
Normalization returns DataFrames instead of writing local intermediate files.


In [ ]:
raw_reviews, raw_items = download_data()
print({"downloaded_reviews": len(raw_reviews), "downloaded_items": len(raw_items)})


### Transform in memory

transform_data returns reviews, users, and items. It validates inputs and creates
review features, user aggregates, and item metadata/aggregates. The recency
reference defaults to the latest supplied review. Nothing is written locally or
to Supabase in this cell. Inspect the counts before running publication.


In [ ]:
reviews, users, items = transform_data(raw_reviews, raw_items)
print({"reviews": len(reviews), "users": len(users), "items": len(items),
       "reference_time": reviews.attrs["as_of"]})


### Replace the three Supabase data tables

The function first checks nonempty inputs, unique IDs, review references, and
JSON serialization. Missing/non-finite values are converted to JSON null.
It then deletes Review rows before parent User/Item rows, inserts users and items,
and finally inserts reviews in batches. The deletion includes legacy null-ID rows.
Errors stop the operation and propagate; there is no automatic rollback.

Running this cell is the intentional publication action. No separate upload
script or local file export is required.


In [ ]:
published_rows = replace_data_in_supabase(reviews, users, items)
print({"published_rows": published_rows})


### Next: prepare experiments from the published source

Run train_test_split_data.ipynb again after replacing the tables, then use
cf_recommend.ipynb or cb_recommend.ipynb. Existing local experiment files are not
automatically refreshed. Training and publishing model bundles remain separate
manual actions; replacing data does not update deployed models.
